In [10]:
import os
import torch
import pandas as pd
from PIL import Image
from transformers import pipeline
from sklearn.metrics import (
    log_loss, accuracy_score,
    precision_score, recall_score, confusion_matrix
)

BASE_DIR = "/Users/kaatsandoval/code/katiarojas87/final-project"


In [2]:
df_cleaned = pd.read_pickle('assessment_output_v2.pkl')
df_cleaned

,source_id,listing_url,image_url,RoomType,Luxury,Brightness,Condition,image_name,image_path,default_image,room_type,scoring_dict,score_lux,score_bright,score_cond
0,20277732,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,https://suumo.jp/front/gazo/bukken/030/N010000...,kitchen,yes,yes,new,20277732_eaa711a75a.jpg,raw_data/suumo_images/20277732/20277732_eaa711...,1,kitchen,"{'luxury': 0.37890625, 'brightness': 0.2695312...",0.378906,0.269531,0.562500
1,20277732,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,https://suumo.jp/front/gazo/bukken/030/N010000...,living room,yes,yes,new,20277732_7c57695877.jpg,raw_data/suumo_images/20277732/20277732_7c5769...,1,floor plan,"{'luxury': -1000, 'brightness': -1000, 'condit...",-1000.000000,-1000.000000,-1000.000000
2,20277732,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,https://suumo.jp/front/gazo/bukken/030/N010000...,living room,yes,yes,new,20277732_274be42c2b.jpg,raw_data/suumo_images/20277732/20277732_274be4...,1,bathroom,"{'luxury': 0.37890625, 'brightness': 0.2226562...",0.378906,0.222656,0.437500
3,20277732,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,https://suumo.jp/front/gazo/bukken/030/N010000...,kitchen,yes,yes,new,20277732_c35b746f90.jpg,raw_data/suumo_images/20277732/20277732_c35b74...,1,kitchen,"{'luxury': 0.4375, 'brightness': 0.18359375, '...",0.437500,0.183594,0.500000
4,20277732,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,https://suumo.jp/front/gazo/bukken/030/N010000...,bedroom,no,yes,old,20277732_bbd7ffee05.jpg,raw_data/suumo_images/20277732/20277732_bbd7ff...,1,bathroom,"{'luxury': 0.53125, 'brightness': 0.1328125, '...",0.531250,0.132812,0.468750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272,77620806,https://suumo.jp/ms/chuko/tokyo/sc_edogawa/nc_...,https://suumo.jp/front/gazo/bukken/030/N010000...,bedroom,yes,no,new,77620806_71029a73b3.jpg,raw_data/suumo_images/77620806/77620806_71029a...,1,bedroom,"{'luxury': 0.203125, 'brightness': 0.0859375, ...",0.203125,0.085938,0.183594
276,20177669,https://suumo.jp/ms/chuko/tokyo/sc_itabashi/nc...,https://suumo.jp/front/gazo/bukken/030/N010000...,bedroom,no,no,new,20177669_41073f369c.jpg,raw_data/suumo_images/20177669/20177669_41073f...,1,bedroom,"{'luxury': 0.1484375, 'brightness': 0.1484375,...",0.148438,0.148438,0.437500
277,20177669,https://suumo.jp/ms/chuko/tokyo/sc_itabashi/nc...,https://suumo.jp/front/gazo/bukken/030/N010000...,kitchen,no,yes,new,20177669_d7adecbe11.jpg,raw_data/suumo_images/20177669/20177669_d7adec...,1,kitchen,"{'luxury': 0.40625, 'brightness': 0.29296875, ...",0.406250,0.292969,0.562500
278,20177669,https://suumo.jp/ms/chuko/tokyo/sc_itabashi/nc...,https://suumo.jp/front/gazo/bukken/030/N010000...,bathroom,yes,yes,new,20177669_d2edcb893f.jpg,raw_data/suumo_images/20177669/20177669_d2edcb...,1,bathroom,"{'luxury': 0.5625, 'brightness': 0.1328125, 'c...",0.562500,0.132812,0.347656


In [14]:
LABEL_TEMPLATES = {
    "luxury": [
        "a luxurious room with premium finishes and elegant furniture",
        "a basic room with cheap furniture and plain finishes",
    ],
    "condition": [
        "a clean, well-maintained room in good condition",
        "a damaged room with peeling paint, stains, or broken fixtures",
    ],
    "brightness": [
        "a bright, well-lit room flooded with light",
        "a dark room with dim lighting",
    ],
}
SCORE_CONFIGS = [
    ("Luxury",     "score_lux",    lambda df: df["Luxury"] == "yes",                    0.55),
    ("Brightness", "score_bright", lambda df: df["Brightness"] == "yes",                0.55),
    ("Condition",  "score_cond",   lambda df: df["Condition"].isin(["new", "yes"]),      0.55),
]


In [ ]:
def build_labels(room_type: str, attribute: str) -> list[str]:
    if attribute not in LABEL_TEMPLATES:
        raise ValueError(f"Unknown attribute '{attribute}'. Choose from: {list(LABEL_TEMPLATES)}")
    return [t.format(room=room_type) for t in LABEL_TEMPLATES[attribute]]


def get_score(image_path: str, labels: list[str], clip_pipeline) -> float:
    image = Image.open(image_path).convert("RGB")
    results = clip_pipeline(image, candidate_labels=labels)
    return {r["label"]: r["score"] for r in results}[labels[0]]


def get_metrics(df: pd.DataFrame) -> dict:
    df = df[df["score_lux"] != -1000].copy()
    results = {}
    for label, score_col, y_true_fn, threshold in SCORE_CONFIGS:
        y_true = y_true_fn(df).astype(int)
        y_prob = df[score_col]
        y_pred = (y_prob > threshold).astype(int)
        results[label] = {
            "log_loss":         log_loss(y_true, y_prob),
            "accuracy":         accuracy_score(y_true, y_pred),
            "precision":        precision_score(y_true, y_pred, zero_division=0),
            "recall":           recall_score(y_true, y_pred, zero_division=0),
            "confusion_matrix": confusion_matrix(y_true, y_pred).ravel().tolist(),
        }
    return results


In [16]:
# ── Main ──────────────────────────────────────────────────────────────────────
df = pd.read_pickle("assessment_output_v2.pkl")
df["image_path"] = df["image_path"].apply(
    lambda p: p if os.path.isabs(p) else os.path.join(BASE_DIR, p)
)

clip = pipeline(
    task="zero-shot-image-classification",
    model="openai/clip-vit-base-patch32",
    device=0 if torch.cuda.is_available() else -1,
)

metrics = get_metrics(df)
for attr, m in metrics.items():
    print(f"\n── {attr.upper()} ──")
    for name, value in m.items():
        print(f"  {name:16s}: {value}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



── LUXURY ──
  log_loss        : 0.6521719025968198
  accuracy        : 0.6267942583732058
  precision       : 0.6590909090909091
  recall          : 0.31521739130434784
  confusion_matrix: [102, 15, 63, 29]

── BRIGHTNESS ──
  log_loss        : 0.8611210456586206
  accuracy        : 0.583732057416268
  precision       : 0.6666666666666666
  recall          : 0.06666666666666667
  confusion_matrix: [116, 3, 84, 6]

── CONDITION ──
  log_loss        : 0.7756063173479119
  accuracy        : 0.4449760765550239
  precision       : 0.7419354838709677
  recall          : 0.3150684931506849
  confusion_matrix: [47, 16, 100, 46]
